<a href="https://colab.research.google.com/github/Aleenapshaji/CaseStudy/blob/main/NLP_CLASSIFICATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Concatenate
from tensorflow.keras.models import Model

import nltk
import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import confusion_matrix


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
!pip install gensim

import gensim


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
#Load dataset
import zipfile

with zipfile.ZipFile(filepath, 'r') as zip_ref:
    zip_ref.extractall('/content/')

df_donor = pd.read_csv('/content/Preprocessed_DonorsChoose_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/DonorChoose.csv'

In [ ]:
filepath = "/content/drive/MyDrive/AI_ML/Data/archive.zip"

#EDA

In [ ]:
df_donor.head()


NameError: name 'df_donor' is not defined

In [ ]:
df_donor.info()


In [ ]:
df_donor.shape()

In [ ]:
df_donor.describe()


In [ ]:
df_donor.isnull().sum()


#Preprocweessing

#Handling missing values

In [ ]:
df_donor['cleaned_titles'] = df_donor['cleaned_titles'].fillna("")


#Duplicates

In [ ]:
df_donor.duplicated().sum()


#Removing punctuations

In [ ]:
string.punctuation


In [ ]:
from nltk.tokenize.sonority_sequencing import punctuation
def remove_punctuation(text):
  punctuationless_text = ''.join([i for i in range in text if i not in string.punctuation])
  return punctuationless_text

#Required columns

In [ ]:
df_donor = df_donor[['cleaned_essays', 'project_is_approved']]
df_donor.head()

#TFIDF

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df_donor['cleaned_essays'])

y = df_donor['project_is_approved']

#Model building

In [ ]:
# spliting data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#Logistic regression

In [ ]:

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_lr))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred_lr))

#Naive byes

In [ ]:
nb = MultinomialNB()

nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_nb))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred_nb))

#Random forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred_rf))

#SVM

In [ ]:
svm = LinearSVC()

svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred_svm))

#COMPARE MODEL

In [ ]:
accuracy = {
    "Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "Naive Bayes": accuracy_score(y_test, y_pred_nb),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
    "SVM": accuracy_score(y_test, y_pred_svm)
}

comparison = pd.DataFrame(
    accuracy.items(),
    columns=["Model", "Accuracy"]
)

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

comparison

#Accuracy bar graph

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(comparison["Model"], comparison["Accuracy"])

plt.title("Model Accuracy Comparison")

plt.xlabel("Models")

plt.ylabel("Accuracy")

plt.xticks(rotation=20)

plt.show()

#Tokenizattion

In [ ]:
sentences = df_donor['cleaned_essays'].apply(word_tokenize)

sentences.head()

#Word2vec model

In [ ]:
from gensim.models import Word2Vec

word2vec_model = Word2Vec(
    sentences,
    vector_size=25,
    window=2,
    min_count=10,
    workers=4,
    epochs=3
)

#Vecctorizzation

In [ ]:
def sentence_vector(words, model):
    vectors = [model.wv[word] for word in words if word in model.wv]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

X_word2vec = np.array(
    [sentence_vector(words, word2vec_model) for words in sentences]
)

y = df_donor['project_is_approved']

In [ ]:
# train test split
X_train_w2v, X_test_w2v, y_train, y_test = train_test_split(
    X_word2vec,
    y,
    test_size=0.2,
    random_state=42
)

#LOGISTIC REGRESSION ON WORD2VEC MODEL

In [ ]:
lr_w2v = LogisticRegression(max_iter=1000)

lr_w2v.fit(X_train_w2v, y_train)

pred = lr_w2v.predict(X_test_w2v)

print("Word2Vec + Logistic Regression Accuracy:", accuracy_score(y_test, pred))

#Final comparison model

In [ ]:
comparison["Word2Vec + Logistic Regression"] = accuracy_score(y_test, pred)

print(comparison)

#Hyper parameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C':[0.01,0.1,1,10]
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=100),
    param_grid,
    cv=3,
    scoring='accuracy'
)

grid_lr.fit(X_train,y_train)

print("Best Parameters:",grid_lr.best_params_)
print("Best Score:",grid_lr.best_score_)

#SVM

In [ ]:
param_grid = {
    'C':[0.1,1,10]
}

grid_svm = GridSearchCV(
    LinearSVC(),
    param_grid,
    cv=3,
    scoring='accuracy'
)

grid_svm.fit(X_train,y_train)

print(grid_svm.best_params_)
print(grid_svm.best_score_)

#ROC curve

In [ ]:
from sklearn.metrics import roc_curve, auc

y_score = lr.predict_proba(X_test)[:,1]

fpr,tpr,_ = roc_curve(y_test,y_score)

roc_auc = auc(fpr,tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr,tpr,label='AUC = %0.2f'%roc_auc)
plt.plot([0,1],[0,1],'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

#Precision recall curve

In [ ]:
from sklearn.metrics import precision_recall_curve

precision,recall,_ = precision_recall_curve(y_test,y_score)

plt.figure(figsize=(6,6))
plt.plot(recall,precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision Recall Curve")
plt.show()

#Tensorflow

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(256,activation='relu',input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64,activation='relu'),

    tf.keras.layers.Dense(1,activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# train
history = model.fit( X_train.toarray(), y_train, validation_split=0.2, epochs=10, batch_size=64 )

In [ ]:
# evaluvate
loss,accuracy = model.evaluate(
    X_test.toarray(),
    y_test
)

print("Test Accuracy:",accuracy)

In [ ]:
# prediction
pred = model.predict(X_test.toarray())

pred = (pred>0.5).astype(int)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test,pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test,pred)

sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(history.history['accuracy'],label='Train Accuracy')
plt.plot(history.history['val_accuracy'],label='Validation Accuracy')

plt.legend()

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.show()

In [ ]:
comparison = pd.DataFrame({

'Model':[
'Logistic Regression',
'SVM',
'Random Forest',
'Deep Learning'
],

'Accuracy':[

accuracy_score(y_test,y_pred_lr),

accuracy_score(y_test,y_pred_svm),

accuracy_score(y_test,y_pred_rf),

accuracy

]

})

comparison

In [ ]:
X_train_tfidf, X_test_tfidf, X_train_w2v, X_test_w2v, y_train, y_test = train_test_split(
    X,
    X_word2vec,
    y,
    test_size=0.2,
    random_state=42
)

X_train_tfidf = X_train_tfidf.toarray()
X_test_tfidf = X_test_tfidf.toarray()

In [ ]:
tfidf_input = Input(shape=(X_train_tfidf.shape[1],), name="TFIDF_Input")

x1 = Dense(256, activation='relu')(tfidf_input)

x1 = Dropout(0.3)(x1)

x1 = Dense(128, activation='relu')(x1)

In [ ]:
w2v_input = Input(shape=(X_train_w2v.shape[1],), name="Word2Vec_Input")

x2 = Dense(128, activation='relu')(w2v_input)

x2 = Dropout(0.3)(x2)

x2 = Dense(64, activation='relu')(x2)

In [ ]:
merged = Concatenate()([x1, x2])

merged = Dense(128, activation='relu')(merged)

merged = Dropout(0.3)(merged)

merged = Dense(64, activation='relu')(merged)

output = Dense(1, activation='sigmoid')(merged)

In [ ]:
model = Model(
    inputs=[tfidf_input, w2v_input],
    outputs=output
)

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    [X_train_tfidf, X_train_w2v],
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64
)

In [ ]:
loss, accuracy = model.evaluate(
    [X_test_tfidf, X_test_w2v],
    y_test
)

print("Test Accuracy:", accuracy)

In [ ]:
pred = model.predict(
    [X_test_tfidf, X_test_w2v]
)

pred = (pred > 0.5).astype(int)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

In [ ]:
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Deep Learning Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")

plt.legend()
plt.show()